In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
import torch
import evaluate
from datasets import load_dataset

In [ ]:
  # import transformers, torch, evaluate, datasets
  # print(transformers.__version__)
  # print(torch.__version__)
  # print(evaluate.__version__)
  # print(datasets.__version__)
  # load this shit as ABOVE CELL DOESNT WORK FOR SOME REASON

In [2]:
# A

# load pre-trained gpt2 model and tokenizer
model_name='gpt2'
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name)

# set up gpu if available
device="cuda" if torch.cuda.is_available() else "cpu"
print(torch.cuda.is_available())
model.to(device)

# surviving first line
prompt_text= "The autumn leaves are falling down,"

# tokenizer analysis
inputs=tokenizer(prompt_text,return_tensors="pt").to(device)
tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Original text: " + prompt_text)
print("Token breakdown: " + tokens.__str__()) # splits words and handles spaces w G symbol

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6666.35it/s]


True
Original text: The autumn leaves are falling down,
Token breakdown: ['The', 'Ġautumn', 'Ġleaves', 'Ġare', 'Ġfalling', 'Ġdown', ',']


In [65]:
# generation and parameter influence

def generate_poem(model, temperature,top_k,top_p):
    outputs=model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,
        min_length=30,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True, # true so u can use temp,topk,topp
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3, # to allow rhymes (which repeat 2 letters not 3 words )
        repetition_penalty=1.2 # punish being repetitive
    )
    return tokenizer.decode(outputs[0],skip_special_tokens=True)

# low temp
print("Temp 0.2 - Predictable and boring")
print(generate_poem(model=model,temperature=0.2, top_k=50,top_p=0.95))
print("-"*30)

# high temp
print("Temp 1.2 - Creative and chaotic")
print(generate_poem(model=model,temperature=1.2, top_k=50,top_p=0.95))
print("-"*30)

# low top_k : restrict vocab to top 10 most likely next words
print("Temp 0.8, Top-K 10 - Focused but less diverse")
print(generate_poem(model=model,temperature=0.8, top_k=10,top_p=1.0))
print("-"*30)

Temp 0.2 - Predictable and boring
The autumn leaves are falling down, and the sun is shining. The sky has changed colour to a bright red; there's no sign of life in it."
"I'm not sure what you're talking about," said Mr. Hirschfeld. "But I think that if we can get some light from this thing then maybe something will happen here somewhere else on Earth where people could see things differently than they do now?"

------------------------------
Temp 1.2 - Creative and chaotic
The autumn leaves are falling down, but the sun is slowly melting in all directions. The air temperature has risen to 80 degrees Kelvin (160 Fahrenheit)—that's close enough that there should be no burning or ice at any moment now either—and this gives you an interesting vantage point between where your brain wants it right before passing a high threshold and here I feel exactly how warm things will become when we stop being cold after just 6 months
------------------------------
Temp 0.8, Top-K 10 - Focused but less

In [42]:
# B

print("loading dataset...")
# load smaller slice of dataset
dataset=load_dataset("merve/poetry",split="train")
dataset_sonnets=dataset.filter(lambda x: x['author'] == 'WILLIAM SHAKESPEARE')

# gpt2 has no padding token so we set it to eos token
tokenizer.pad_token=tokenizer.eos_token

# 1. tokenize dataset
def tokenize_function(examples):
    # dataset has line col so we tokenize it
    return tokenizer(
        examples["content"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

print("tokenizing dataset...")
tokenized_datasets=dataset.map(tokenize_function,batched=True, remove_columns=dataset.column_names)

# 2. data collator
# format data and create labels for clm ( casual language modeling)
data_collator=DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # NO mlm ( masked language modeling )
)

# 3. training arguments
# control how the model learns
training_args=TrainingArguments(
    output_dir="./poetry-gpt2",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    fp16=True, # speed up training on colab gpus
)

# 4. initialize trainer
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator
)

# 5. start fine-tuning
print("starting fine-tuning...")
trainer.train()

# save the fine-tuned model and tokenizer
trainer.save_model("./poetry-gpt2-final")
tokenizer.save_pretrained("./poetry-gpt2-final")
print("model fine-tuned and saved successfully!")

loading dataset...


Repo card metadata block was not found. Setting CardData to empty.


tokenizing dataset...


Map: 100%|██████████| 573/573 [00:00<00:00, 4680.95 examples/s]


starting fine-tuning...


Step,Training Loss
100,4.218654
200,3.560160
300,3.380130


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]

model fine-tuned and saved successfully!


In [45]:
# test the fine-tuned model

# load
model_name='./poetry-gpt2-final'
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name)

# set up gpu if available
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# test the mf
print("Testing the fine-tuned model...")
print(generate_poem(model=model,temperature=0.8,top_k=50,top_p=0.95))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9825.84it/s]


Testing the fine-tuned model...
The autumn leaves are falling down,
And there is only snow-white ground; and when the storm has ceased from a thoroughfare of winding descents we go.  When I see my face brighten by moonlight with our breathless countenance one last time upon this dim earth these hills have taken their place in me: And what shall be done? will not she make up her mind to stay now that thou art


In [9]:
# performance ananlysis

# set up device and tokenizer
device="cuda" if torch.cuda.is_available() else "cpu"
print(f"loading models into : {device.upper()}")

tokenizer=AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token=tokenizer.eos_token

# load hugging face sacreBLEU metric
sacrebleu=evaluate.load("sacrebleu")

# load both models in vram
print("loading model a ( base gpt-2 )")
model_a=AutoModelForCausalLM.from_pretrained("gpt2").to(device)

print("loading model b ( fine-tuned poetry gpt-2 )")
model_b=AutoModelForCausalLM.from_pretrained("./poetry-gpt2-final").to(device)

def generate_continuation(model,prompt_text):
    inputs=tokenizer(prompt_text, return_tensors="pt").to(device)

    outputs=model.generate(
        inputs["input_ids"],
        max_length=40,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

    full_text=tokenizer.decode(outputs[0], skip_special_tokens=True)

    continuation=full_text[len(prompt_text):].strip()
    return continuation

test_cases=[
    {
        "prompt": "The autumn leaves are falling down,\n",
        "reference": "And winter's chill is drawing near."
    },
    {
        "prompt": "A gentle breeze blows through the trees,\n",
        "reference": "It whispers secrets to the leaves."
    },
    {
        "prompt": "The sun sets low beneath the hills,\n",
        "reference": "And the quiet night the valley fills."
    }
]

print("-"*30)
print("Generating performance analysis")

for i,test in enumerate(test_cases,1):
    prompt=test["prompt"]
    reference=test["reference"]

    # run both models
    cont_a=generate_continuation(model_a,prompt)
    cont_b=generate_continuation(model_b,prompt)

    # calc scores
    score_a=sacrebleu.compute(predictions=[cont_a], references=[[reference]])['score']
    score_b=sacrebleu.compute(predictions=[cont_b], references=[[reference]])['score']

    print("-"*30)
    print(f"Test Case {i}:")
    print(f"Prompt:\n{prompt}")
    print(f"Reference:\n{reference}\n")
    print(f"Model A (Base GPT-2) Continuation:\n{cont_a}\nScore: {score_a:.2f}")
    print(f"Model B (Fine-tuned Poetry GPT-2) Continuation:\n{cont_b}\nScore: {score_b:.2f}")
    print("-"*30)

loading models into : CUDA
loading model a ( base gpt-2 )


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10904.63it/s]


loading model b ( fine-tuned poetry gpt-2 )


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10855.62it/s]


------------------------------
Generating performance analysis
------------------------------
Test Case 1:
Prompt:
The autumn leaves are falling down,

Reference:
And winter's chill is drawing near.

Model A (Base GPT-2) Continuation:
R.N.O.: 1/20th season of "Dumbbells" with Michael Bay's The Devil in the Details: a must-read
Score: 1.44
Model B (Fine-tuned Poetry GPT-2) Continuation:
 lucy laughter in deep voice;� and there's no moving to listen out for ÃÂÃÂtill
Score: 1.79
------------------------------
------------------------------
Test Case 2:
Prompt:
A gentle breeze blows through the trees,

Reference:
It whispers secrets to the leaves.

Model A (Base GPT-2) Continuation:
(And when she heard that) a voice came up to her. She was about eight years old, and had always been very shy with him, but
Score: 1.43
Model B (Fine-tuned Poetry GPT-2) Continuation:
 So you see she is dressed as a guest.  She walks alone and takes no care of us; but
Score: 1.29
------------------------------
